# ROLEFIELD: Social Curation Field Mapping

This notebook demonstrates the field mapping pipeline for identifying the European
innovation intermediary field through Twitter list co-classification.

## Methodology

Following Schiffer et al.'s social curation approach:
1. **Seed selection**: Core EIT/KIC accounts
2. **Network discovery**: Two-ring expansion via list co-memberships
3. **Denoising**: Random walk centrality filtering
4. **Network merging**: Combine seed networks
5. **Community detection**: Louvain algorithm
6. **Content analysis**: TF-IDF on bios

In [ ]:
import json
import sys
sys.path.insert(0, '../src')

from rolebox_social import (
    CoCurationNetwork, ListCollector,
    FieldMapper, FieldNetwork,
    BridgingAnalyzer, TriangulationAnalyzer,
)

## 1. Load Analysis Results

Load the pre-computed field analysis from the full corpus.

In [ ]:
with open('../data/outputs/field_analysis.json', 'r') as f:
    field_results = json.load(f)

print(f"Analyzed {field_results['network_statistics']['n_actors']:,} actors")
print(f"Relations: {field_results['network_statistics']['n_relations']:,}")
print(f"Sub-regions identified: {len(field_results['sub_regions'])}")

## 2. Network Statistics

Key properties of the merged co-classification network.

In [ ]:
stats = field_results['network_statistics']

print("=== Network Statistics ===")
print(f"Actors: {stats['n_actors']:,}")
print(f"Relations: {stats['n_relations']:,}")
print(f"Density: {stats['density']:.4f}")
print(f"Modularity: {stats['modularity']:.2f}")
print(f"Avg Degree: {stats['avg_degree']:.1f}")
print(f"Components: {stats['n_components']} (single connected component)")
print(f"\nLists analyzed: {stats['total_lists_analyzed']:,}")
print(f"Unique curators: {stats['total_curators']:,}")

## 3. Seed Accounts

The eight EIT/KIC accounts used to discover the field.

In [ ]:
print("=== Seed Accounts ===")
print(f"{'Account':<20} {'Type':<12} {'Followers':>10} {'Lists':>8} {'Ring1':>8} {'Ring2':>8}")
print("="*70)

for seed in field_results['seeds']:
    print(f"{seed['username']:<20} {seed['type']:<12} {seed['followers']:>10,} {seed['lists']:>8} {seed['ring1_size']:>8,} {seed['ring2_size']:>8,}")

## 4. Sub-Region Structure

Ten sub-regions identified through Louvain community detection.

In [ ]:
print("=== Sub-Regions ===")
print(f"{'ID':<4} {'Label':<30} {'Size':>8} {'Share':>8} {'Density':>8}")
print("="*62)

for sr in field_results['sub_regions']:
    print(f"{sr['id']:<4} {sr['label']:<30} {sr['size']:>8,} {sr['field_share']*100:>7.1f}% {sr['internal_density']:>8.2f}")

print("\n=== Characteristic Vocabulary (Top 3 per region) ===")
for sr in field_results['sub_regions'][:5]:
    terms = ', '.join(sr['top_terms'][:3])
    print(f"{sr['label']}: {terms}")

## 5. Actor Heterogeneity

Distribution of actor types in the field.

In [ ]:
print("=== Actor Type Distribution ===")
print(f"{'Type':<25} {'Count':>10} {'Percentage':>12}")
print("="*50)

for atype, data in field_results['actor_types'].items():
    label = atype.replace('_', ' ').title()
    print(f"{label:<25} {data['count']:>10,} {data['percentage']:>11.0f}%")

print("\n→ Key insight: Field includes diverse actor types")
print("  Individuals (33%) + Startups (22%) = 55% non-institutional actors")

## 6. KIC Positioning

Where do KICs sit within the externally-perceived field structure?

In [ ]:
print("=== KIC Positioning ===")
print("="*70)

for kic, pos in field_results['kic_positioning'].items():
    print(f"\n{kic}")
    print(f"  Primary region: {pos['primary_region']}")
    print(f"  Bridging score: {pos['bridging_score']:.2f}")
    print(f"  Regions connected: {len(pos['regions_connected'])}")
    print(f"  → {pos['interpretation']}")

## 7. Triangulation Analysis

Load triangulation results mapping internal sociotypes to external sub-regions.

In [ ]:
with open('../data/outputs/triangulation_analysis.json', 'r') as f:
    triang = json.load(f)

print("=== Sociotype → Sub-Region Triangulation ===")
print("="*80)

for t in triang['triangulation']:
    print(f"\n{t['sociotype'].upper()}")
    print(f"  Internal: {t['internal_character']}")
    print(f"  External: {', '.join(t['corresponding_regions'])}")
    print(f"  Alignment: {t['alignment'].upper()}")
    print(f"  → {t['interpretation']}")

## 8. Key Observations

In [ ]:
print("=== KEY OBSERVATIONS ===")
for i, obs in enumerate(triang['key_observations'], 1):
    print(f"\n{i}. {obs}")

print("\n" + "="*70)
print(f"VALIDATION: {triang['validation_summary']['overall_validation']}")
print(f"  Strong alignments: {triang['validation_summary']['strong_alignments']}")
print(f"  No alignment (expected): {triang['validation_summary']['no_alignment']}")

## 9. Bridging Analysis

In [ ]:
bridging = triang['bridging_analysis']

print("=== Bridging Analysis ===")
print(f"Bridging actors: {bridging['n_bridging_actors']}")
print(f"Cross-region ties: {bridging['total_cross_region_ties']:,}")
print(f"Avg bridging score: {bridging['avg_bridging_score']:.2f}")

print("\nTop Bridging Actors:")
for actor in bridging['top_bridging_actors']:
    connects = ', '.join(actor['connects']) if isinstance(actor['connects'], list) else actor['connects']
    print(f"  {actor['username']}: {actor['bridging_score']:.2f} - {connects}")

print("\nHigh-bridging actor types:")
for atype, pct in bridging['high_bridging_actor_types'].items():
    print(f"  {atype.replace('_', ' ').title()}: {pct*100:.0f}%")

## 10. Conclusion

In [ ]:
print("=== CONCLUSION ===")
print("")
print("External classification through social curation VALIDATES internal")
print("sociotype configurations identified from KIC self-presentations.")
print("")
print("Key findings:")
print("1. KICs' interstitial character is externally recognized")
print("2. Thematic organization dominates field structure")
print("3. Connective function ≠ distinct sub-region (function vs domain)")
print("4. EU Policy forms distinct sub-region (accountability validated)")
print("5. KICs occupy INTERFACING positions - bridging while anchoring")